# Auditor Opinion Forecasting: Financial-Only and Prior-Opinion Models (v3)

Owner: Jessica Bedson (Tasks 1.1 to 1.6).

Changes from v2, made in response to team review:

1. **Seed (decision-log item 6).** The seed is read from `config.SEED` (20260928), not hard-coded as 42. Every random step uses it: model fitting, calibration and bootstrap intervals.
2. **Selection rule applied, not asserted.** The primary model is whichever financial-only model has the highest validation ROC-AUC. v2 asserted that XGBoost would win; v3 applies the rule and names the artifacts after the winner.
3. **Benchmark corrected.** The model card compares the financial-only model with the corrected 0.70 to 0.75 benchmark, not 0.826.
4. **Artifact names consistent.** Model files, the model card and the reproducibility section are all named from the same version IDs.
5. **Scoring module.** `scoring.py` contains only `score_firm_core`. The contract surface (session alias, percentile, cohort base rate, `uses_prior_opinion`, the three-key envelope) belongs to the adapter in `src/model.py` on main, which calls `score_firm_core`.
6. **Clean run.** Run with *Restart Kernel and Run All* so every output comes from one top-to-bottom run (needed for Task 7.2).

## Setup

In [ ]:
import hashlib
import importlib
import json
import pickle
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Seed and data path come from the shared project config so that Task 7.2
# (clean-machine regeneration) reproduces every number.
DECISION_LOG_SEED = 20260928
try:
    import config
except ImportError:
    config = None

if config is not None and hasattr(config, "SEED"):
    RANDOM_SEED = int(config.SEED)
    SEED_SOURCE = "config.SEED"
else:
    RANDOM_SEED = DECISION_LOG_SEED
    SEED_SOURCE = "decision-log value (config.SEED not importable from this folder)"
assert RANDOM_SEED == DECISION_LOG_SEED, (
    f"Seed {RANDOM_SEED} differs from the decision log ({DECISION_LOG_SEED}). "
    "Amend the decision log or fix config before freezing."
)

DATA_PATH = Path(getattr(config, "DATA_PATH", "final_panel 1.csv"))
N_BOOTSTRAP = 1000
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

FEATURE_COLS = [
    "LNEMP", "LNTA", "LNSALE", "LNTI", "IntanTA", "InvCA", "CapInt",
    "TobinQ", "RET1Y", "ROA", "ROE", "GPM", "CURR_RATIO", "AT_TURN",
    "INV_TURN", "DPR", "PTB", "PE", "cfsale",
]
PRIOR_COL = "prior_adverse_opinion"
FINANCIAL_ONLY = list(FEATURE_COLS)
WITH_PRIOR = FEATURE_COLS + [PRIOR_COL]

DEV_YEARS = (2011, 2017)
TRAIN_YEARS = (2011, 2015)
VAL_YEARS = (2016, 2017)
TEST_YEARS = (2018, 2021)
TEST_OUTCOME_YEARS = [2019, 2020, 2021, 2022]

# Corrected benchmark for the financial-only model (team decision, replaces 0.826).
BENCHMARK_FIN_ONLY = (0.70, 0.75)

# Selection rule, fixed before test evaluation.
FINANCIAL_CANDIDATES = ["Logistic regression", "Random Forest", "XGBoost"]
ALGO_SHORT = {"Logistic regression": "lr", "Random Forest": "rf", "XGBoost": "xgb"}

print("Random seed:", RANDOM_SEED, f"({SEED_SOURCE}) | bootstrap draws:", N_BOOTSTRAP)
print("Data file:", DATA_PATH)
print("Versions: python", platform.python_version(), "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)

In [ ]:
def prepare_features(frame):
    """Task 1.3 rule: cfsale = oibdp / sale, zero sales treated as missing; infinite values become missing."""
    out = frame.copy()
    sales = pd.to_numeric(out["sale"], errors="raise")
    income = pd.to_numeric(out["oibdp"], errors="raise")
    out["cfsale"] = income / sales.replace(0, np.nan)
    out[FEATURE_COLS] = (
        out[FEATURE_COLS].apply(pd.to_numeric, errors="raise")
        .replace([np.inf, -np.inf], np.nan)
    )
    return out


def log_odds(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


def pr_auc(y, p):
    precision, recall, _ = precision_recall_curve(y, p)
    return auc(recall, precision)


def metric_row(model, subgroup, y, p):
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    return {
        "model": model,
        "subgroup": subgroup,
        "rows": int(len(y)),
        "adverse": int(y.sum()),
        "observed_rate": float(y.mean()),
        "mean_predicted": float(p.mean()),
        "ROC_AUC": roc_auc_score(y, p),
        "PR_AUC": pr_auc(y, p),
        "Brier": brier_score_loss(y, p),
    }


def bootstrap_ci(y, p, n=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    roc, pra = [], []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        yb = y[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        roc.append(roc_auc_score(yb, p[idx]))
        pra.append(pr_auc(yb, p[idx]))
    return {
        "ROC_lo": np.percentile(roc, 2.5), "ROC_hi": np.percentile(roc, 97.5),
        "PR_lo": np.percentile(pra, 2.5), "PR_hi": np.percentile(pra, 97.5),
    }


def bootstrap_gap(y, p_base, p_new, n=N_BOOTSTRAP, seed=RANDOM_SEED):
    """Gap = new model minus base model, on identical bootstrap resamples."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    p_base = np.asarray(p_base, dtype=float)
    p_new = np.asarray(p_new, dtype=float)
    roc, pra = [], []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        yb = y[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        roc.append(roc_auc_score(yb, p_new[idx]) - roc_auc_score(yb, p_base[idx]))
        pra.append(pr_auc(yb, p_new[idx]) - pr_auc(yb, p_base[idx]))
    return {
        "ROC_gap": roc_auc_score(y, p_new) - roc_auc_score(y, p_base),
        "ROC_lo": np.percentile(roc, 2.5), "ROC_hi": np.percentile(roc, 97.5),
        "PR_gap": pr_auc(y, p_new) - pr_auc(y, p_base),
        "PR_lo": np.percentile(pra, 2.5), "PR_hi": np.percentile(pra, 97.5),
    }


def show(df, digits=4):
    print(df.round(digits).to_string(index=False))

## Load the panel

In [ ]:
raw = pd.read_csv(
    DATA_PATH,
    dtype={"gvkey": "string", "cik": "string", "cusip": "string", "tic": "string"},
    parse_dates=["datadate", "INFO_DATE"],
)

## Task 1.1: Label filter

Only assessed firm-years are kept: auopic 1 (effective internal controls) and 2 (adverse opinion). Code 0, codes 3 and 4, and missing labels are excluded and counted so the filter reconciles to the full panel.

In [ ]:
print("Loaded:", f"{len(raw):,}", "rows |", f"{raw.gvkey.nunique():,}", "firms | FY",
      raw.fyear.min(), "-", raw.fyear.max())
print("SHA256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())

assert raw[["gvkey", "fyear"]].notna().all().all(), "Missing firm identifiers or fiscal years."
assert not raw.duplicated(["gvkey", "fyear"]).any(), "Duplicate firm-year observations."

print("\nOriginal auopic counts (including missing):")
print(raw["auopic"].value_counts(dropna=False).sort_index().to_string())

assessed = raw.loc[raw["auopic"].isin([1, 2])].copy()
exclusions = {
    "Code 0 (unknown or SOX exempt)": int(raw["auopic"].eq(0).sum()),
    "Codes 3 and 4": int(raw["auopic"].isin([3, 4]).sum()),
    "Missing label": int(raw["auopic"].isna().sum()),
}
assert len(assessed) + sum(exclusions.values()) == len(raw), "Exclusions do not reconcile."

print("\nExcluded rows:")
for k, v in exclusions.items():
    print(f"  {k}: {v:,}")
n_adverse_assessed = int(assessed["auopic"].eq(2).sum())
print(f"\nRetained firm-years: {len(assessed):,} (effective {len(assessed) - n_adverse_assessed:,}, "
      f"adverse {n_adverse_assessed:,})")
print(f"Adverse rate among assessed firm-years: {n_adverse_assessed / len(assessed):.2%}")

## Task 1.2: Next-year outcome pairs

Fiscal-year t features predict the fiscal-year t+1 opinion. A pair is kept only when the same firm has an assessed opinion in exactly year t+1. The prediction date is the feature year's INFO_DATE. Nothing observed after year t is used as a feature: the prior-opinion indicator is the year-t opinion, which is known at the prediction date.

In [ ]:
assessed["target"] = (assessed["auopic"] == 2).astype(int)
assessed["feature_year"] = assessed["fyear"].astype(int)
assessed["prediction_date"] = assessed["INFO_DATE"]
assessed[PRIOR_COL] = assessed["target"]   # year-t opinion, used as a feature for year t+1

outcomes = assessed[["gvkey", "feature_year", "INFO_DATE", "target"]].rename(columns={
    "feature_year": "outcome_year",
    "INFO_DATE": "outcome_info_date",
    "target": "outcome_target",
})

pairs = assessed.copy()
pairs["outcome_year"] = pairs["feature_year"] + 1
pairs = pairs.merge(outcomes, on=["gvkey", "outcome_year"], how="inner", validate="one_to_one")
pairs["target"] = pairs["outcome_target"].astype(int)
pairs = pairs.sort_values(["prediction_date", "gvkey"]).reset_index(drop=True)

assert (pairs["outcome_year"] == pairs["feature_year"] + 1).all(), "Outcome is not exactly t+1."
assert pairs["prediction_date"].lt(pairs["outcome_info_date"]).all(), "Prediction date must precede outcome date."
assert not pairs.duplicated(["gvkey", "feature_year"]).any()

unmatched = len(assessed) - len(pairs)
print(f"Assessed feature-year rows: {len(assessed):,}")
print(f"Without an exact next-year assessed opinion: {unmatched:,}")
print(f"Retained consecutive-year pairs: {len(pairs):,} across {pairs.gvkey.nunique():,} firms")
print(f"Next-year adverse outcomes: {int(pairs.target.sum()):,} ({pairs.target.mean():.2%})")

## Task 1.3: Feature table

The 19 financial predictors are built from the original unsuffixed columns. cfsale is oibdp divided by sale, with zero sales treated as missing.

In [ ]:
pairs = prepare_features(pairs)

n_zero_sales = int(pd.to_numeric(pairs["sale"]).eq(0).sum())
n_finite_cfsale = int(pairs["cfsale"].notna().sum())
print(f"Zero-sales pairs: {n_zero_sales:,}")
print(f"Finite cfsale: {n_finite_cfsale:,} / {len(pairs):,} ({n_finite_cfsale / len(pairs):.2%})")

missingness = pd.DataFrame({
    "missing_rows": pairs[FEATURE_COLS].isna().sum(),
    "missing_percent": pairs[FEATURE_COLS].isna().mean() * 100,
}).sort_values("missing_percent", ascending=False)
print("\nMissingness before imputation (pooled, pre-split):")
print(missingness.round(2).to_string())

## Task 1.4: Temporal split (Enoch Banda, reviewed)

This follows Enoch's revised split. Feature year t predicts outcome year t+1.

In [ ]:
development = pairs.loc[pairs["feature_year"].between(*DEV_YEARS)]
train_candidates = development.loc[development["feature_year"].between(*TRAIN_YEARS)].copy()
val_candidates = development.loc[development["feature_year"].between(*VAL_YEARS)].copy()
test_candidates = pairs.loc[pairs["feature_year"].between(*TEST_YEARS)].copy()

val_start = val_candidates["prediction_date"].min()
test_start = test_candidates["prediction_date"].min()

X_train = (train_candidates.loc[train_candidates["outcome_info_date"] < val_start]
           .sort_values(["prediction_date", "gvkey"]).reset_index(drop=True))
X_val = (val_candidates.loc[val_candidates["outcome_info_date"] < test_start]
         .sort_values(["prediction_date", "gvkey"]).reset_index(drop=True))
X_test = test_candidates.sort_values(["prediction_date", "gvkey"]).reset_index(drop=True)

for name, part in [("Train", X_train), ("Validation", X_val), ("Test", X_test)]:
    assert (part["outcome_year"] == part["feature_year"] + 1).all(), f"{name}: t+1 check failed."
    assert part["target"].nunique() == 2, f"{name}: both classes are required."

assert X_train["outcome_info_date"].max() < X_val["prediction_date"].min(), "Train labels overlap validation."
assert X_val["outcome_info_date"].max() < X_test["prediction_date"].min(), "Validation labels overlap test."
assert set(X_train["feature_year"]).isdisjoint(X_val["feature_year"])
assert set(X_val["feature_year"]).isdisjoint(X_test["feature_year"])

def split_summary(part, name):
    return {
        "partition": name,
        "rows": int(len(part)),
        "firms": int(part["gvkey"].nunique()),
        "feature_years": f"{int(part.feature_year.min())}-{int(part.feature_year.max())}",
        "outcome_years": f"{int(part.outcome_year.min())}-{int(part.outcome_year.max())}",
        "prediction_dates": f"{part.prediction_date.min().date()} to {part.prediction_date.max().date()}",
        "adverse": int(part["target"].sum()),
        "adverse_rate": float(part["target"].mean()),
    }

split_table = pd.DataFrame([
    split_summary(X_train, "Training"),
    split_summary(X_val, "Validation"),
    split_summary(X_test, "Test"),
])
print(f"Timing exclusions: training {len(train_candidates) - len(X_train):,}, "
      f"validation {len(val_candidates) - len(X_val):,}")
show(split_table)
split_table.to_csv(RESULTS_DIR / "split_summary_v3.csv", index=False)

split_record = {
    "design": "Feature year t predicts outcome year t+1 (auopic = 2)",
    "source": "Enoch Banda, Task 1.4, reviewed",
    "timing_rule": "Earlier partition outcome INFO_DATE must precede the next partition's earliest prediction date",
    "info_date_limitation": "INFO_DATE is an availability proxy, not a verified auditor report date",
    "partitions": split_table.to_dict("records"),
}

## Task 1.5A: Model fitting on training data

Six models are fitted on training data only: logistic regression, Random Forest and XGBoost, each on the financial predictors alone and with the prior-opinion indicator added. Each pipeline fits median imputation with missing indicators on training data. The logistic regression also standardizes its inputs. Hyperparameters are fixed before validation results are seen. The logistic-regression prior-opinion variant is included so the selection rule has a matching variant whichever financial-only model wins.

In [ ]:
def make_lr():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, class_weight=None, random_state=RANDOM_SEED)),
    ])

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("classifier", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10, max_features="sqrt",
            class_weight=None, random_state=RANDOM_SEED, n_jobs=-1)),
    ])

def make_xgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("classifier", XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            random_state=RANDOM_SEED, n_jobs=2)),
    ])

MODEL_DESCRIPTIONS = {
    "Logistic regression": "Logistic regression (standardized inputs, max_iter 5000)",
    "Random Forest": "Random Forest (300 trees, minimum samples per leaf 10, sqrt features per split)",
    "XGBoost": "XGBoost (300 trees, maximum depth 3, learning rate 0.05, minimum child weight 5, subsampling 0.8)",
}

SPECS = {
    "Logistic regression": (make_lr, FINANCIAL_ONLY),
    "Random Forest": (make_rf, FINANCIAL_ONLY),
    "XGBoost": (make_xgb, FINANCIAL_ONLY),
    "Logistic regression + prior opinion": (make_lr, WITH_PRIOR),
    "Random Forest + prior opinion": (make_rf, WITH_PRIOR),
    "XGBoost + prior opinion": (make_xgb, WITH_PRIOR),
}

y_train = X_train["target"].to_numpy()
y_val = X_val["target"].to_numpy()
y_test = X_test["target"].to_numpy()

fitted = {}
for name, (builder, cols) in SPECS.items():
    fitted[name] = (builder().fit(X_train[cols], y_train), cols)
print("Fitted:", ", ".join(fitted))

## Task 1.5B: Validation comparison and model selection

Pre-specified rule: the primary model is the financial-only model with the highest validation ROC-AUC among logistic regression, Random Forest and XGBoost. The variant is the same algorithm with the prior-opinion indicator. The rule is applied in code, so if the seed change moves the winner, the artifacts follow the rule.

In [ ]:
val_scores = {name: fitted[name][0].predict_proba(X_val[fitted[name][1]])[:, 1] for name in SPECS}
val_table = pd.DataFrame([metric_row(n, "Validation", y_val, val_scores[n]) for n in SPECS])
show(val_table)

val_auc = {n: roc_auc_score(y_val, val_scores[n]) for n in FINANCIAL_CANDIDATES}
ranked = sorted(FINANCIAL_CANDIDATES, key=val_auc.get, reverse=True)
PRIMARY, RUNNER_UP = ranked[0], ranked[1]
VARIANT = f"{PRIMARY} + prior opinion"
PRIMARY_VERSION = f"financial_only_{ALGO_SHORT[PRIMARY]}_v3"
VARIANT_VERSION = f"prior_opinion_{ALGO_SHORT[PRIMARY]}_v3"

gap_runner_up = bootstrap_gap(y_val, val_scores[RUNNER_UP], val_scores[PRIMARY])
print(f"\nPrimary (highest validation ROC-AUC among financial-only models): {PRIMARY}")
print(f"Variant: {VARIANT}")
print(f"{PRIMARY} minus {RUNNER_UP} on validation: ROC {gap_runner_up['ROC_gap']:.4f} "
      f"[{gap_runner_up['ROC_lo']:.4f}, {gap_runner_up['ROC_hi']:.4f}]")
if gap_runner_up["ROC_lo"] < 0 < gap_runner_up["ROC_hi"]:
    print("Note: the interval includes zero, so the two models are not distinguishable on validation. "
          "The pre-specified rule still decides.")
print(f"Artifact versions: {PRIMARY_VERSION}, {VARIANT_VERSION}")

selection_record = {
    "rule": "Highest validation ROC-AUC among financial-only logistic regression, Random Forest, XGBoost",
    "validation_roc_auc": {k: float(v) for k, v in val_auc.items()},
    "primary": PRIMARY,
    "runner_up": RUNNER_UP,
    "primary_minus_runner_up": {k: float(v) for k, v in gap_runner_up.items()},
    "variant": VARIANT,
}
val_table.to_csv(RESULTS_DIR / "validation_metrics_v3.csv", index=False)

## Task 1.5C: Calibration (fitted on validation only)

In [ ]:
calibrators = {}
for name in [PRIMARY, VARIANT]:
    cal = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED)
    cal.fit(log_odds(val_scores[name]), y_val)
    assert cal.coef_[0, 0] > 0, f"{name}: calibration reversed the ranking."
    calibrators[name] = cal

for name in [PRIMARY, VARIANT]:
    cal_val = calibrators[name].predict_proba(log_odds(val_scores[name]))[:, 1]
    print(f"{name}:")
    print(f"  observed validation rate {y_val.mean():.2%} | mean raw {val_scores[name].mean():.2%} "
          f"| mean calibrated {cal_val.mean():.2%}")
    print(f"  Brier raw {brier_score_loss(y_val, val_scores[name]):.5f} "
          f"| Brier calibrated {brier_score_loss(y_val, cal_val):.5f}")

fig, ax = plt.subplots(figsize=(6, 5))
for name in [PRIMARY, VARIANT]:
    cal_val = calibrators[name].predict_proba(log_odds(val_scores[name]))[:, 1]
    obs, pred = calibration_curve(y_val, cal_val, n_bins=5, strategy="quantile")
    ax.plot(pred, obs, marker="o", label=f"{name} (calibrated)")
ax.plot([0, 0.25], [0, 0.25], "--", color="gray", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed adverse rate")
ax.set_title("Validation calibration (fitting diagnostic)")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "validation_calibration_v3.png", dpi=200)
plt.show()

## Task 1.5D: Test evaluation (pooled and by outcome year)

In [ ]:
test_scores = {name: fitted[name][0].predict_proba(X_test[fitted[name][1]])[:, 1] for name in SPECS}
test_cal = {name: calibrators[name].predict_proba(log_odds(test_scores[name]))[:, 1] for name in [PRIMARY, VARIANT]}

POOLED = f"Pooled {TEST_OUTCOME_YEARS[0]}-{TEST_OUTCOME_YEARS[-1]}"
groups = [(POOLED, np.ones(len(X_test), dtype=bool))]
groups += [(f"Outcome {y}", X_test["outcome_year"].to_numpy() == y) for y in TEST_OUTCOME_YEARS]

rows = []
for label, mask in groups:
    yy = y_test[mask]
    for name in SPECS:
        rows.append(metric_row(f"{name} (raw)", label, yy, test_scores[name][mask]))
    for name in [PRIMARY, VARIANT]:
        rows.append(metric_row(f"{name} (calibrated)", label, yy, test_cal[name][mask]))
    constant = np.full(mask.sum(), y_train.mean())
    rows.append({"model": "Constant prevalence benchmark", "subgroup": label,
                 "rows": int(mask.sum()), "adverse": int(yy.sum()),
                 "observed_rate": float(yy.mean()), "mean_predicted": float(constant.mean()),
                 "ROC_AUC": np.nan, "PR_AUC": np.nan,
                 "Brier": brier_score_loss(yy, constant)})

test_table = pd.DataFrame(rows)
show(test_table)
test_table.to_csv(RESULTS_DIR / "test_metrics_v3.csv", index=False)

ci_rows, gap_rows = [], []
for label, mask in groups:
    yy = y_test[mask]
    for name in [PRIMARY, VARIANT]:
        b = bootstrap_ci(yy, test_cal[name][mask])
        ci_rows.append({"model": name, "subgroup": label, **b})
    g = bootstrap_gap(yy, test_cal[PRIMARY][mask], test_cal[VARIANT][mask])
    gap_rows.append({"subgroup": label, **g})

ci_table = pd.DataFrame(ci_rows)
gap_table = pd.DataFrame(gap_rows)
print("\nBootstrap 95% intervals (calibrated probabilities):")
show(ci_table)
print("Gap: prior-opinion model minus primary model (calibrated):")
show(gap_table)
ci_table.to_csv(RESULTS_DIR / "test_bootstrap_ci_v3.csv", index=False)
gap_table.to_csv(RESULTS_DIR / "test_model_gap_v3.csv", index=False)

### Per-year ROC-AUC chart

In [ ]:
labels = [g[0] for g in groups]
x = np.arange(len(labels))
width = 0.38
fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, name in [(-width / 2, PRIMARY), (width / 2, VARIANT)]:
    sub = ci_table[ci_table.model == name].set_index("subgroup").loc[labels]
    est = test_table[test_table.model == f"{name} (calibrated)"].set_index("subgroup").loc[labels]["ROC_AUC"]
    yerr = np.vstack([est.values - sub["ROC_lo"].values, sub["ROC_hi"].values - est.values])
    ax.bar(x + offset, est.values, width, label=name, yerr=yerr, capsize=3)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.axhspan(*BENCHMARK_FIN_ONLY, color="gray", alpha=0.12, label="Financial-only benchmark 0.70-0.75")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("ROC-AUC (test)")
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "test_roc_by_outcome_year_v3.png", dpi=200)
plt.show()

## Task 1.5E: Sensitivity check, relaxed non-overlapping split

In [ ]:
relaxed_train = train_candidates
relaxed_val = val_candidates
y_rtrain = relaxed_train["target"].to_numpy()
y_rval = relaxed_val["target"].to_numpy()

relaxed_test = {}
for name in [PRIMARY, VARIANT]:
    builder, cols = SPECS[name]
    model = builder().fit(relaxed_train[cols], y_rtrain)
    v_raw = model.predict_proba(relaxed_val[cols])[:, 1]
    cal = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED)
    cal.fit(log_odds(v_raw), y_rval)
    t_raw = model.predict_proba(X_test[cols])[:, 1]
    relaxed_test[name] = cal.predict_proba(log_odds(t_raw))[:, 1]

print(f"Relaxed training rows: {len(relaxed_train):,} (buffered: {len(X_train):,})")
print(f"Relaxed validation rows: {len(relaxed_val):,}, adverse {int(y_rval.sum())} "
      f"(buffered: {len(X_val):,}, adverse {int(y_val.sum())})")
print(f"Rows restored by relaxing the buffer: "
      f"{(len(relaxed_train) - len(X_train)) + (len(relaxed_val) - len(X_val)):,}")

sens_rows = []
for name in [PRIMARY, VARIANT]:
    sens_rows.append(metric_row(f"{name} (relaxed, calibrated)", POOLED, y_test, relaxed_test[name]))
    sens_rows.append(metric_row(f"{name} (primary split, calibrated)", POOLED, y_test, test_cal[name]))
sens_table = pd.DataFrame(sens_rows)
show(sens_table)
relaxed_gap = bootstrap_gap(y_test, relaxed_test[PRIMARY], relaxed_test[VARIANT])
print("\nRelaxed split gap (prior minus primary):",
      f"ROC {relaxed_gap['ROC_gap']:.4f} [{relaxed_gap['ROC_lo']:.4f}, {relaxed_gap['ROC_hi']:.4f}],",
      f"PR {relaxed_gap['PR_gap']:.4f} [{relaxed_gap['PR_lo']:.4f}, {relaxed_gap['PR_hi']:.4f}]")
sens_table.to_csv(RESULTS_DIR / "sensitivity_relaxed_split_v3.csv", index=False)

## Task 1.6A: Freeze the models and verify reloading

In [ ]:
FROZEN_AT = datetime.now(timezone.utc).isoformat()
SOURCE_SHA = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

def freeze(name, version_id, cols):
    model = fitted[name][0]
    artifact = {
        "model_version": version_id,
        "model_name": name,
        "pipeline": model,
        "calibrator": calibrators[name],
        "feature_cols": list(cols),
        "prior_opinion_col": PRIOR_COL if PRIOR_COL in cols else None,
        "default_probability": "calibrated",
        "calibration_transform": {"method": "sigmoid applied to log-odds", "clip_min": 1e-6, "clip_max": 1 - 1e-6},
        "target_definition": "Fiscal-year t+1 auopic equals 2",
        "split_definition": split_record,
        "selection": selection_record,
        "test_pooled": test_table[test_table.subgroup == POOLED].to_dict("records"),
        "source_file": DATA_PATH.name,
        "source_sha256": SOURCE_SHA,
        "random_seed": RANDOM_SEED,
        "seed_source": SEED_SOURCE,
        "frozen_at_utc": FROZEN_AT,
        "versions": {
            "python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn.__version__, "xgboost": xgboost.__version__,
        },
    }
    path = MODEL_DIR / f"{version_id}.pkl"
    with path.open("wb") as f:
        pickle.dump(artifact, f, protocol=pickle.HIGHEST_PROTOCOL)
    return path

primary_path = freeze(PRIMARY, PRIMARY_VERSION, FINANCIAL_ONLY)
variant_path = freeze(VARIANT, VARIANT_VERSION, WITH_PRIOR)

for path, name in [(primary_path, PRIMARY), (variant_path, VARIANT)]:
    with path.open("rb") as f:
        art = pickle.load(f)
    raw_p = art["pipeline"].predict_proba(X_test[art["feature_cols"]])[:, 1]
    cal_p = art["calibrator"].predict_proba(log_odds(raw_p))[:, 1]
    # XGBoost stores trees in float32, so a reloaded model can differ by about 1e-9 absolute.
    np.testing.assert_allclose(raw_p, test_scores[name], rtol=1e-6, atol=1e-8)
    np.testing.assert_allclose(cal_p, test_cal[name], rtol=1e-6, atol=1e-8)
    print(f"{art['model_version']}: reloaded predictions match test evaluation "
          f"({len(art['feature_cols'])} predictors) -> {path.name}")

decisions = {
    "split": split_record,
    "selection": selection_record,
    "primary_version": PRIMARY_VERSION,
    "variant_version": VARIANT_VERSION,
    "random_seed": RANDOM_SEED,
    "seed_source": SEED_SOURCE,
    "benchmark_financial_only_roc_auc": list(BENCHMARK_FIN_ONLY),
    "frozen_at_utc": FROZEN_AT,
    "source_sha256": SOURCE_SHA,
}
(MODEL_DIR / "decisions_v3.json").write_text(json.dumps(decisions, indent=2, default=str), encoding="utf-8")
print("Decision record written:", MODEL_DIR / "decisions_v3.json")

## Task 1.6B: Model card

In [ ]:
def md_table(df, columns):
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for _, r in df.iterrows():
        cells = []
        for c in columns:
            v = r[c]
            cells.append(f"{v:.4f}" if isinstance(v, (float, np.floating)) and not np.isnan(v) else
                         ("n/a" if isinstance(v, (float, np.floating)) else str(v)))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

def ci_txt(model, subgroup):
    r = ci_table[(ci_table.model == model) & (ci_table.subgroup == subgroup)].iloc[0]
    return f"{r.ROC_lo:.3f} to {r.ROC_hi:.3f}"

def pooled_roc(name):
    sel = (test_table.model == f"{name} (calibrated)") & (test_table.subgroup == POOLED)
    return test_table.loc[sel, "ROC_AUC"].iloc[0]

pooled_primary_roc = pooled_roc(PRIMARY)
pooled_variant_roc = pooled_roc(VARIANT)
lo, hi = BENCHMARK_FIN_ONLY
if pooled_primary_roc < lo:
    benchmark_status = f"below the corrected financial-only benchmark of {lo:.2f} to {hi:.2f}"
elif pooled_primary_roc > hi:
    benchmark_status = f"above the corrected financial-only benchmark of {lo:.2f} to {hi:.2f}"
else:
    benchmark_status = f"within the corrected financial-only benchmark of {lo:.2f} to {hi:.2f}"

split_md = md_table(split_table, ["partition", "rows", "firms", "feature_years", "outcome_years", "prediction_dates", "adverse", "adverse_rate"])
val_md = md_table(val_table, ["model", "rows", "adverse", "observed_rate", "mean_predicted", "ROC_AUC", "PR_AUC", "Brier"])
test_md = md_table(test_table, ["model", "subgroup", "rows", "adverse", "observed_rate", "mean_predicted", "ROC_AUC", "PR_AUC", "Brier"])
gap_md = md_table(gap_table, ["subgroup", "ROC_gap", "ROC_lo", "ROC_hi", "PR_gap", "PR_lo", "PR_hi"])
sens_md = md_table(sens_table, ["model", "rows", "adverse", "mean_predicted", "ROC_AUC", "PR_AUC", "Brier"])
g_ru = selection_record["primary_minus_runner_up"]

ENV_JSON = json.dumps({"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                       "scikit_learn": sklearn.__version__, "xgboost": xgboost.__version__})
CARD_PATH = MODEL_DIR / f"{PRIMARY_VERSION}_model_card.md"

card = f"""# Auditor Opinion Forecast Models v3

Frozen at UTC: {FROZEN_AT}

Status: capstone models built on the reviewed temporal split, frozen for evaluation. The primary (financial-only) model's pooled test ROC-AUC is {pooled_primary_roc:.3f} (95% bootstrap interval {ci_txt(PRIMARY, POOLED)}), {benchmark_status}. The prior-opinion variant's pooled test ROC-AUC is {pooled_variant_roc:.3f} (95% interval {ci_txt(VARIANT, POOLED)}). ROC-AUC measures ranking ability; it is not the share of correct predictions.

## Prediction task

Use fiscal-year t information to forecast whether a firm receives an adverse internal-control opinion (auopic = 2) in fiscal year t+1. Both years are restricted to auopic codes 1 and 2. This is an internal-control risk indicator, not a fraud finding. No feature uses information observed after year t.

## Models

- Primary ({PRIMARY_VERSION}): {MODEL_DESCRIPTIONS[PRIMARY]} on 19 financial predictors.
- Variant ({VARIANT_VERSION}): the same {PRIMARY} with the fiscal-year t adverse-opinion indicator added.
- Selection rule, fixed before test evaluation: highest validation ROC-AUC among logistic regression, Random Forest and XGBoost on financial predictors. {PRIMARY} was selected; the runner-up was {RUNNER_UP} (validation ROC-AUC difference {g_ru['ROC_gap']:.4f}, 95% interval {g_ru['ROC_lo']:.4f} to {g_ru['ROC_hi']:.4f}).
- Comparators, reported on test: the other financial-only models and their prior-opinion variants.
- Preprocessing: median imputation with missing indicators, fitted on training data only.
- Calibration: sigmoid mapping of log-odds, fitted on validation data only.

## Temporal split

{split_md}

Timing rule: an earlier partition's outcome INFO_DATE must precede the next partition's earliest prediction date. INFO_DATE is an availability proxy, not a verified report date.

## Validation (model selection)

{val_md}

Validation calibration is a fitting diagnostic. It uses {int(y_val.sum())} adverse cases and is not independent evaluation.

## Test results (outcome years {TEST_OUTCOME_YEARS[0]} to {TEST_OUTCOME_YEARS[-1]})

{test_md}

Bootstrap 95% interval for the primary model's pooled ROC-AUC: {ci_txt(PRIMARY, POOLED)}.
Bootstrap 95% interval for the variant's pooled ROC-AUC: {ci_txt(VARIANT, POOLED)}.

## Gap between the two models (prior-opinion variant minus primary, calibrated)

{gap_md}

## Sensitivity: relaxed non-overlapping split

The timing buffer is removed. Training has {len(relaxed_train):,} rows and validation has {len(relaxed_val):,} rows with {int(y_rval.sum())} adverse cases. The test set is unchanged. Training labels in this version can be known after some validation prediction dates, so this is a robustness check only.

{sens_md}

## Use by the agent (score_firm)

The adapter in src/model.py owns the tool contract. The agent calls score_firm with a session alias such as FIRM_A; the adapter resolves the alias inside the local tool layer and calls score_firm_core(gvkey, feature_year, panel, artifact) from scoring.py. score_firm_core returns the probability, model version, rank among assessed firms in the feature year, and the names of any imputed features. It returns no raw financial values. It does return the real gvkey, so it is called only inside the local tool layer, never directly by the agent. A risk score is not proof of a future adverse opinion.

## Limitations

- INFO_DATE is an availability proxy. The auditor report publication date is not verified.
- The test window (outcomes {TEST_OUTCOME_YEARS[0]} to {TEST_OUTCOME_YEARS[-1]}) overlaps the period examined in the opinion-history experiment, so test results are not fully clean.
- Validation has {int(y_val.sum())} adverse cases, so calibration is unstable.
- Per-year test samples are small, so the per-year intervals are wide.
- Firms recur across periods, so the test measures future firm-years rather than unseen firms only.
- The prior-opinion variant mostly reflects persistence of control weaknesses. It should be presented as such, and the report should state how much the financial features add beyond it.
- No operating threshold is selected. The score_firm rank is retrospective, not an as-of-date live ranking.
- Until the university confirms the WRDS data-use terms, agent tests use dummy data only.

## Reproducibility

Source file: {DATA_PATH.name}
Source SHA256: {SOURCE_SHA}
Random seed: {RANDOM_SEED} ({SEED_SOURCE})
Bootstrap draws: {N_BOOTSTRAP}
Environment: {ENV_JSON}
Frozen artifacts: {primary_path.as_posix()} and {variant_path.as_posix()}
Decision record: {(MODEL_DIR / "decisions_v3.json").as_posix()}
"""
CARD_PATH.write_text(card, encoding="utf-8")
print("Model card written:", CARD_PATH)
print(f"Pooled test ROC-AUC, primary {pooled_primary_roc:.4f} ({benchmark_status}), variant {pooled_variant_roc:.4f}")

## Task 1.6C: Scoring module

`scoring.py` contains one public function:

- `score_firm_core(gvkey, feature_year, panel, artifact)` scores one firm-year with a frozen model and ranks it among assessed firms in the same feature year. It uses only feature-year data and returns no raw financial values.

The tool contract is owned by the adapter in `src/model.py` on main (team decision, Mark, Nov 10). The adapter takes the contract signature, resolves the session alias, calls `score_firm_core`, converts rank to percentile, adds the cohort base rate and `uses_prior_opinion`, and returns the three-key envelope. `score_firm_core` returns the real gvkey, so only the adapter calls it.

In [ ]:
SCORING_SRC = r"""
import numpy as np
import pandas as pd

FEATURE_COLS = [
    "LNEMP", "LNTA", "LNSALE", "LNTI", "IntanTA", "InvCA", "CapInt",
    "TobinQ", "RET1Y", "ROA", "ROE", "GPM", "CURR_RATIO", "AT_TURN",
    "INV_TURN", "DPR", "PTB", "PE", "cfsale",
]



def _normalize_gvkey(value):
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text.lstrip("0") or "0"


def _prepare(frame):
    out = frame.copy()
    sales = pd.to_numeric(out["sale"], errors="raise")
    income = pd.to_numeric(out["oibdp"], errors="raise")
    out["cfsale"] = income / sales.replace(0, np.nan)
    out[FEATURE_COLS] = (
        out[FEATURE_COLS].apply(pd.to_numeric, errors="raise")
        .replace([np.inf, -np.inf], np.nan)
    )
    return out


def score_firm_core(gvkey, feature_year, panel, artifact):
    # Score one firm-year with a frozen model.
    # Called only by the adapter in src/model.py, which owns the tool contract and the alias.
    # Forecast: adverse internal-control opinion (auopic = 2) in fiscal year t+1.
    # Rank: descending risk among assessed firms in feature year t (1 = highest risk).
    # Uses only data from feature year t.
    year = int(feature_year)
    cols = list(artifact["feature_cols"])
    prior_col = artifact.get("prior_opinion_col")

    cohort = panel.loc[panel["fyear"].eq(year) & panel["auopic"].isin([1, 2])].copy()
    if cohort.empty:
        raise ValueError(f"No assessed firms available for FY{year}.")
    if cohort["gvkey"].isna().any():
        raise ValueError("Missing firm identifiers in the cohort.")

    cohort["_key"] = cohort["gvkey"].map(_normalize_gvkey)
    if cohort["_key"].duplicated().any():
        raise ValueError("Duplicate firm-year identifiers.")
    selected = cohort["_key"].eq(_normalize_gvkey(gvkey))
    if selected.sum() != 1:
        raise ValueError(
            f"Expected one assessed record for this firm in FY{year}; found {int(selected.sum())}."
        )

    cohort = _prepare(cohort)
    if prior_col:
        cohort[prior_col] = (cohort["auopic"] == 2).astype(int)
    features = cohort[cols].apply(pd.to_numeric, errors="raise")

    raw = artifact["pipeline"].predict_proba(features)[:, 1]
    settings = artifact["calibration_transform"]
    clipped = np.clip(raw, settings["clip_min"], settings["clip_max"])
    calibrated = artifact["calibrator"].predict_proba(
        np.log(clipped / (1 - clipped)).reshape(-1, 1)
    )[:, 1]

    version = artifact["default_probability"]
    if version not in {"original", "calibrated"}:
        raise ValueError("Unknown default probability version.")
    probability = calibrated if version == "calibrated" else raw

    ranks = pd.Series(probability).rank(ascending=False, method="min").astype(int).to_numpy()
    position = int(np.flatnonzero(selected.to_numpy())[0])
    row = cohort.iloc[position]

    return {
        "gvkey": str(row["gvkey"]),
        "feature_year": year,
        "outcome_year": year + 1,
        "prediction_date": pd.Timestamp(row["INFO_DATE"]).isoformat(),
        "model_version": artifact["model_version"],
        "probability": float(probability[position]),
        "probability_version": version,
        "probability_original": float(raw[position]),
        "probability_calibrated": float(calibrated[position]),
        "risk_rank": int(ranks[position]),
        "cohort_size": int(len(cohort)),
        "rank_definition": "Descending risk among assessed firms in the feature year; 1 is highest; ties use minimum rank.",
        "missing_features_imputed": features.columns[features.iloc[position].isna()].tolist(),
        "interpretation": "Retrospective score from the frozen model. The year-wide rank is not an as-of-date live ranking. A risk score is not proof of a future adverse opinion.",
    }

"""
Path("scoring.py").write_text(SCORING_SRC.lstrip(), encoding="utf-8")
print("Wrote scoring.py")

## Task 1.6D: Scoring verification

Two checks: `score_firm_core` reproduces the test-set predictions for both frozen models, and its output contains no raw financial values. The example saved to `results/` drops the gvkey, which only the adapter sees.

In [ ]:
import scoring
importlib.reload(scoring)

rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(X_test), size=6, replace=False)
checked = []
for path in [primary_path, variant_path]:
    with path.open("rb") as f:
        art = pickle.load(f)
    name = art["model_name"]
    for i in sample_idx:
        row = X_test.iloc[int(i)]
        res = scoring.score_firm_core(row["gvkey"], row["feature_year"], raw, art)
        np.testing.assert_allclose(res["probability_original"], test_scores[name][int(i)], rtol=1e-6, atol=1e-8)
        np.testing.assert_allclose(res["probability_calibrated"], test_cal[name][int(i)], rtol=1e-6, atol=1e-8)
        checked.append({"model": art["model_version"], "test_row": int(i), "feature_year": res["feature_year"],
                        "probability": res["probability"], "risk_rank": res["risk_rank"],
                        "cohort_size": res["cohort_size"]})
show(pd.DataFrame(checked), digits=6)
print("\nCheck 1 passed: score_firm_core matches the test evaluation for both models.")

primary_art = pickle.loads(primary_path.read_bytes())
demo_row = X_test.iloc[int(sample_idx[0])]
example = scoring.score_firm_core(demo_row["gvkey"], demo_row["feature_year"], raw, primary_art)
assert not set(example) & set(FEATURE_COLS), "Raw feature values must not be returned."
print("Check 2 passed: no raw feature values in the output.")

example.pop("gvkey")   # the real identifier stays inside the local tool layer
(RESULTS_DIR / "score_firm_example_v3.json").write_text(json.dumps(example, indent=2), encoding="utf-8")
print("\nscore_firm_core output (gvkey removed for the saved example):")
print(json.dumps(example, indent=2))

## Run summary

In [ ]:
summary = {
    "seed": f"{RANDOM_SEED} ({SEED_SOURCE})",
    "primary_model": f"{PRIMARY} -> {primary_path.as_posix()}",
    "variant_model": f"{VARIANT} -> {variant_path.as_posix()}",
    "primary_pooled_test_roc_auc": f"{pooled_primary_roc:.4f} [{ci_txt(PRIMARY, POOLED)}], {benchmark_status}",
    "variant_pooled_test_roc_auc": f"{pooled_variant_roc:.4f} [{ci_txt(VARIANT, POOLED)}]",
    "model_card": CARD_PATH.as_posix(),
    "scoring_module": "scoring.py (score_firm_core; called by the adapter in src/model.py)",
    "score_example": (RESULTS_DIR / "score_firm_example_v3.json").as_posix(),
}
for k, v in summary.items():
    print(f"{k:30s} {v}")